# PM2.5 Forecasting — GBT Spark v10 (Hybrid)

**Kết hợp điểm mạnh của từng version, chọn params tốt nhất từng K:**

| K range | Strategy | Nguồn |
|---|---|---|
| K=1–22 | params v8: `minInst↑, subSample↓, stepSize↓` + `FEATURES_SHORT/MED` | v8 thắng 7/8 K |
| K=25 | params v5: `minInst=10, subSample=0.75` + `FEATURES_LONG` | v5 best tại K=25 |
| K=28–49 | params v9: `minInst=8, subSample=0.75` + `FEATURES_MED` | v9 tốt hơn v8 6/8 K |

**Logic `get_gbt_params` và `get_base_features` hoàn toàn per-K** — không có ranh giới cứng.

In [ ]:
# ─── CELL 0: SETUP ────────────────────────────────────────────────
!apt-get install -y openjdk-11-jdk -q
!pip install pyspark==3.4.1 pyarrow scipy -q

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, lag, lead, avg, stddev, when, hour, month,
    dayofweek, sin, cos, to_timestamp
)
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
import numpy as np
import pandas as pd
import json, os

spark = SparkSession.builder \
    .appName('PM25_GBT_v10') \
    .config('spark.driver.memory', '6g') \
    .config('spark.driver.extraJavaOptions', '-Xss128m') \
    .config('spark.executor.extraJavaOptions', '-Xss128m') \
    .config('spark.sql.execution.arrow.pyspark.enabled', 'false') \
    .getOrCreate()

print('✅ Spark:', spark.version)

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjd

In [ ]:
# ─── CELL 1: LOAD & CLEAN ─────────────────────────────────────────
FC_COLS = (
    [f'temp_fc{x}'     for x in [1,3,6,12,24,48,72]] +
    [f'wind_fc{x}'     for x in [1,3,6,12,24,48,72]] +
    [f'winddir_fc{x}'  for x in [1,3,6,12,24,48,72]] +
    [f'blh_fc{x}'      for x in [1,3,6,12,24,48,72]] +
    [f'humid_fc{x}'    for x in [1,3,6,12,24,48,72]] +
    [f'precip_fc{x}'   for x in [1,3,6,12,24,48,72]] +
    [f'pressure_fc{x}' for x in [1,3,6,12,24,48,72]]
)

df = spark.read.csv('/content/weather_aqi_hourly.csv', header=True, inferSchema=True)
df = df \
    .withColumn('ts',    to_timestamp(col('Local_Time'))) \
    .withColumn('hour',  hour('ts')) \
    .withColumn('month', month('ts')) \
    .drop('UTC_Time','City','Country_Code','Timezone',
          'AQI','AQI_Pollutant', *FC_COLS) \
    .filter(col('PM25').isNotNull()) \
    .orderBy('ts')

df.cache()
print(f'✅ Rows: {df.count():,}')
print(f'BLH null: {df.filter(col("BLH").isNull()).count():,} rows')

✅ Rows: 32,585
BLH null: 4,368 rows


In [ ]:
# ─── CELL 2: BLH IMPUTATION ───────────────────────────────────────
from pyspark.sql.functions import percentile_approx

blh_median = df.filter(col('BLH').isNotNull()) \
    .groupBy('hour','month') \
    .agg(percentile_approx('BLH', 0.5).alias('blh_med'))

df = df.join(blh_median, on=['hour','month'], how='left') \
       .withColumn('BLH', when(col('BLH').isNull(), col('blh_med')).otherwise(col('BLH'))) \
       .drop('blh_med')

still_null = df.filter(col('BLH').isNull()).count()
if still_null > 0:
    g_med = df.filter(col('BLH').isNotNull()) \
        .agg(percentile_approx('BLH', 0.5).alias('g')).collect()[0]['g']
    df = df.withColumn('BLH', when(col('BLH').isNull(), lit(g_med)).otherwise(col('BLH')))
    print(f'  Fallback global median={g_med:.1f} cho {still_null} rows')

print(f'✅ BLH null sau impute: {df.filter(col("BLH").isNull()).count()}')
df.cache()

✅ BLH null sau impute: 0


DataFrame[hour: int, month: int, Local_Time: timestamp, CO: double, NO2: double, O3: double, PM10: double, PM25: double, SO2: double, Clouds: int, Precipitation: double, Pressure: double, Relative_Humidity: int, Temperature: double, Wind_Speed: double, Wind_Direction: int, BLH: double, ts: timestamp]

In [ ]:
# ─── CELL 3: FEATURE ENGINEERING ──────────────────────────────────
from pyspark.sql.functions import datediff, to_date

w_ts   = Window.orderBy('ts')
w_r8   = Window.orderBy('ts').rowsBetween(-8,     -1)
w_r24  = Window.orderBy('ts').rowsBetween(-24,    -1)
w_r48  = Window.orderBy('ts').rowsBetween(-48,    -1)
w_r72  = Window.orderBy('ts').rowsBetween(-72,    -1)
w_r168 = Window.orderBy('ts').rowsBetween(-168,   -1)
w_365  = Window.orderBy('ts').rowsBetween(-365*24,-1)

df = df.withColumn('pm25_current', col('PM25'))

df = df \
    .withColumn('pm25_lag1',   lag('PM25',   1).over(w_ts)) \
    .withColumn('pm25_lag6',   lag('PM25',   6).over(w_ts)) \
    .withColumn('pm25_lag24',  lag('PM25',  24).over(w_ts)) \
    .withColumn('pm25_lag48',  lag('PM25',  48).over(w_ts)) \
    .withColumn('pm25_lag72',  lag('PM25',  72).over(w_ts)) \
    .withColumn('pm25_lag168', lag('PM25', 168).over(w_ts))

df = df \
    .withColumn('pm25_roll8_mean',   avg('PM25').over(w_r8)) \
    .withColumn('pm25_roll8_std',    stddev('PM25').over(w_r8)) \
    .withColumn('pm25_roll8_max',    F.max('PM25').over(w_r8)) \
    .withColumn('pm25_roll8_min',    F.min('PM25').over(w_r8)) \
    .withColumn('pm25_roll24_mean',  avg('PM25').over(w_r24)) \
    .withColumn('pm25_roll24_std',   stddev('PM25').over(w_r24)) \
    .withColumn('pm25_roll24_max',   F.max('PM25').over(w_r24)) \
    .withColumn('pm25_roll48_mean',  avg('PM25').over(w_r48)) \
    .withColumn('pm25_roll48_std',   stddev('PM25').over(w_r48)) \
    .withColumn('pm25_roll48_max',   F.max('PM25').over(w_r48)) \
    .withColumn('pm25_roll72_mean',  avg('PM25').over(w_r72)) \
    .withColumn('pm25_roll168_mean', avg('PM25').over(w_r168)) \
    .withColumn('pm25_roll168_std',  stddev('PM25').over(w_r168))

df = df \
    .withColumn('pm25_diff_1_6',  col('pm25_lag1') - col('pm25_lag6')) \
    .withColumn('pm25_diff_6_24', col('pm25_lag6') - col('pm25_lag24')) \
    .withColumn('pm25_accel',
        (col('pm25_lag1') - col('pm25_lag6')) -
        (col('pm25_lag6') - col('pm25_lag24'))) \
    .withColumn('pm25_trend_24h',  col('pm25_lag1') - col('pm25_lag24')) \
    .withColumn('pm25_anomaly_score',
        (col('pm25_lag1') - col('pm25_roll24_mean')) /
        (col('pm25_roll24_std') + lit(1e-6))) \
    .withColumn('is_spike', (col('pm25_anomaly_score') > lit(2.0)).cast('int')) \
    .withColumn('pm25_norm',
        col('pm25_lag1') / (col('pm25_roll24_mean') + lit(1e-6))) \
    .withColumn('pm25_range_24',
        col('pm25_roll24_max') - col('pm25_roll24_mean'))

df = df \
    .withColumn('pm25_roll365_mean', avg('PM25').over(w_365)) \
    .withColumn('pm25_level_ratio',
        col('pm25_roll24_mean') / (col('pm25_roll365_mean') + lit(1e-6))) \
    .withColumn('days_since_start',
        datediff(to_date(col('ts')), to_date(lit('2022-08-11'))).cast('float'))

df = df \
    .withColumn('is_rainy_season', col('month').between(5,10).cast('int')) \
    .withColumn('is_burn_season',  col('month').isin([4,5,6,10,11,12]).cast('int')) \
    .withColumn('is_april',        (col('month') == 4).cast('int')) \
    .withColumn('is_rush_hour',    col('hour').isin([7,8,9,17,18,19]).cast('int')) \
    .withColumn('is_night',        ((col('hour') >= 22) | (col('hour') <= 5)).cast('int')) \
    .withColumn('is_weekend',      dayofweek(col('ts')).isin([1,7]).cast('int')) \
    .withColumn('hour_sin',        sin(col('hour')  * lit(2*3.14159265/24))) \
    .withColumn('hour_cos',        cos(col('hour')  * lit(2*3.14159265/24))) \
    .withColumn('month_sin',       sin(col('month') * lit(2*3.14159265/12))) \
    .withColumn('month_cos',       cos(col('month') * lit(2*3.14159265/12)))

df = df \
    .withColumn('wind_dir_sin', sin(col('Wind_Direction') * lit(3.14159265/180.0))) \
    .withColumn('wind_dir_cos', cos(col('Wind_Direction') * lit(3.14159265/180.0)))

df = df \
    .withColumn('temperature_lag1',       lag('Temperature',       1).over(w_ts)) \
    .withColumn('wind_speed_lag1',        lag('Wind_Speed',        1).over(w_ts)) \
    .withColumn('relative_humidity_lag1', lag('Relative_Humidity', 1).over(w_ts)) \
    .withColumn('pressure_lag1',          lag('Pressure',          1).over(w_ts)) \
    .withColumn('precip_lag1',            lag('Precipitation',     1).over(w_ts)) \
    .withColumn('precip_lag6',            lag('Precipitation',     6).over(w_ts)) \
    .withColumn('precip_lag24',           lag('Precipitation',    24).over(w_ts)) \
    .withColumn('blh_lag1',               lag('BLH',               1).over(w_ts)) \
    .withColumn('blh_lag6',               lag('BLH',               6).over(w_ts)) \
    .withColumn('blh_roll24_mean',        avg('BLH').over(w_r24)) \
    .withColumn('clouds_lag1',            lag('Clouds',            1).over(w_ts))

df = df \
    .withColumn('co_lag1',   lag('CO',   1).over(w_ts)) \
    .withColumn('co_lag6',   lag('CO',   6).over(w_ts)) \
    .withColumn('no2_lag1',  lag('NO2',  1).over(w_ts)) \
    .withColumn('no2_lag6',  lag('NO2',  6).over(w_ts)) \
    .withColumn('so2_lag1',  lag('SO2',  1).over(w_ts)) \
    .withColumn('pm10_lag1', lag('PM10', 1).over(w_ts)) \
    .withColumn('pm10_lag6', lag('PM10', 6).over(w_ts)) \
    .withColumn('o3_lag1',   lag('O3',   1).over(w_ts))

df = df \
    .withColumn('wind_x_humidity',   col('wind_speed_lag1') * col('relative_humidity_lag1')) \
    .withColumn('temp_x_humidity',   col('temperature_lag1') * col('relative_humidity_lag1')) \
    .withColumn('blh_x_wind',        col('blh_lag1') * col('wind_speed_lag1')) \
    .withColumn('pm10_pm25_ratio',   col('pm10_lag1') / (col('pm25_lag1') + lit(1e-6))) \
    .withColumn('dispersion_idx',    col('wind_speed_lag1') * (lit(100) - col('relative_humidity_lag1'))) \
    .withColumn('blh_pm25_interact', col('pm25_lag1') / (col('blh_lag1') + lit(1.0)))

w_r6acc  = Window.orderBy('ts').rowsBetween(-6,  -1)
w_r12acc = Window.orderBy('ts').rowsBetween(-12, -1)
w_r24acc = Window.orderBy('ts').rowsBetween(-24, -1)

df = df \
    .withColumn('rain_acc_6h',  F.sum('Precipitation').over(w_r6acc)) \
    .withColumn('rain_acc_12h', F.sum('Precipitation').over(w_r12acc)) \
    .withColumn('rain_acc_24h', F.sum('Precipitation').over(w_r24acc)) \
    .withColumn('rain_washout', (F.sum('Precipitation').over(w_r6acc) > lit(1.0)).cast('int'))

df = df \
    .withColumn('rained_lag1',  (lag('Precipitation',  1).over(w_ts) > lit(0.1)).cast('int')) \
    .withColumn('rained_lag3',  (lag('Precipitation',  3).over(w_ts) > lit(0.1)).cast('int')) \
    .withColumn('rained_lag6',  (lag('Precipitation',  6).over(w_ts) > lit(0.1)).cast('int')) \
    .withColumn('rained_lag12', (lag('Precipitation', 12).over(w_ts) > lit(0.1)).cast('int')) \
    .withColumn('rained_lag24', (lag('Precipitation', 24).over(w_ts) > lit(0.1)).cast('int'))

df = df.withColumn('hours_since_rain',
    when(col('Precipitation') > lit(0.1),  lit(0.0))
    .when(col('rained_lag1')  == lit(1),   lit(1.0))
    .when(col('rained_lag3')  == lit(1),   lit(3.0))
    .when(col('rained_lag6')  == lit(1),   lit(6.0))
    .when(col('rained_lag12') == lit(1),   lit(12.0))
    .when(col('rained_lag24') == lit(1),   lit(24.0))
    .otherwise(lit(48.0))
).drop('rained_lag1','rained_lag3','rained_lag6','rained_lag12','rained_lag24')

df = df \
    .withColumn('wind_rainy_interact', col('wind_speed_lag1') * col('is_rainy_season')) \
    .withColumn('blh_rainy_interact',  col('blh_lag1') * col('is_rainy_season')) \
    .withColumn('precip_trend',        col('precip_lag1') - col('precip_lag6'))

df.cache()
print('✅ Feature engineering done')

✅ Feature engineering done


In [ ]:
# ─── CELL 4: SPLIT + FEATURE LISTS ───────────────────────────────
CUTOFF_VAL  = '2025-07-01'
CUTOFF_TEST = '2025-11-01'

train_df_check = df.filter(col('ts') < lit(CUTOFF_VAL))
val_df_check   = df.filter((col('ts') >= lit(CUTOFF_VAL)) & (col('ts') < lit(CUTOFF_TEST)))
test_df_check  = df.filter(col('ts') >= lit(CUTOFF_TEST))
print(f'Train: {train_df_check.count():,} | Val: {val_df_check.count():,} | Test: {test_df_check.count():,}')

WEATHER_COLS_LEAD = [
    'Temperature','Wind_Speed','Relative_Humidity',
    'Pressure','Precipitation','BLH',
    'wind_dir_sin','wind_dir_cos'
]
WEATHER_AT_K = [f'{c}_atK' for c in WEATHER_COLS_LEAD]

RAIN_FEATS = [
    'hours_since_rain',
    'rain_acc_6h','rain_acc_12h','rain_acc_24h',
    'rain_washout','is_rainy_season',
    'wind_rainy_interact','blh_rainy_interact','precip_trend',
]

FEATURES_SHORT = [
    'pm25_current','pm25_lag1','pm25_lag6',
    'pm25_roll8_mean','pm25_roll8_std','pm25_roll8_max','pm25_roll8_min',
    'pm25_diff_1_6','pm25_accel','pm25_anomaly_score','is_spike','pm25_norm',
    'pm25_range_24',
    'temperature_lag1','wind_speed_lag1','relative_humidity_lag1',
    'pressure_lag1','precip_lag1','dispersion_idx','blh_pm25_interact',
    'blh_lag1','blh_lag6','blh_roll24_mean',
    'wind_dir_sin','wind_dir_cos',
    'no2_lag1','pm10_lag1','co_lag1','so2_lag1',
    'hour_sin','hour_cos','is_rush_hour','is_night','is_weekend',
    'is_burn_season','is_april',
    'pm25_lag24','pm25_roll24_mean','pm25_roll24_std',
    'pm25_lagK',
] + RAIN_FEATS

FEATURES_MED = [
    'pm25_lag1','pm25_lag6','pm25_lag24',
    'pm25_roll8_mean','pm25_roll24_mean','pm25_roll48_mean',
    'pm25_roll8_std','pm25_roll24_std','pm25_roll8_max','pm25_roll24_max',
    'pm25_diff_1_6','pm25_diff_6_24','pm25_trend_24h',
    'pm25_anomaly_score','pm25_norm','is_spike',
    'pm25_range_24','pm25_roll168_mean','pm25_roll168_std',
    'pm25_roll365_mean','pm25_level_ratio','days_since_start',
    'temperature_lag1','wind_speed_lag1','relative_humidity_lag1',
    'pressure_lag1','precip_lag1','precip_lag6',
    'blh_lag1','blh_lag6','blh_roll24_mean','blh_x_wind',
    'wind_dir_sin','wind_dir_cos','dispersion_idx','blh_pm25_interact',
    'clouds_lag1','no2_lag1','no2_lag6','pm10_lag1','pm10_lag6',
    'co_lag1','co_lag6','so2_lag1','o3_lag1',
    'wind_x_humidity','temp_x_humidity','pm10_pm25_ratio',
    'hour_sin','hour_cos','month_sin','month_cos',
    'is_rush_hour','is_night','is_weekend','is_burn_season','is_april',
    'pm25_current','pm25_lag48',
    'pm25_lagK',
] + RAIN_FEATS

# FEATURES_LONG: giữ nguyên từ v5, CHỈ dùng cho K=25
FEATURES_LONG = [
    'pm25_lag1','pm25_lag6',
    'pm25_lag24','pm25_lag48','pm25_lag72','pm25_lag168',
    'pm25_roll24_mean','pm25_roll48_mean','pm25_roll72_mean',
    'pm25_roll24_std','pm25_roll48_std','pm25_roll24_max','pm25_roll48_max',
    'pm25_diff_6_24','pm25_trend_24h','pm25_anomaly_score','pm25_norm',
    'pm25_range_24','pm25_roll168_mean','pm25_roll168_std',
    'pm25_roll365_mean','pm25_level_ratio','days_since_start',
    'temperature_lag1','wind_speed_lag1','relative_humidity_lag1',
    'pressure_lag1','precip_lag6','precip_lag24',
    'blh_lag1','blh_lag6','blh_roll24_mean',
    'wind_dir_sin','wind_dir_cos','dispersion_idx',
    'no2_lag1','pm10_lag1','co_lag1',
    'hour_sin','hour_cos','month_sin','month_cos',
    'is_weekend','is_night','is_burn_season','is_april',
    'pm25_lagK',
] + RAIN_FEATS

def get_base_features(K):
    """
    [v10 hybrid] Feature list per K:
    - K<=8:  SHORT (ngắn hạn)
    - K<=24: MED   (trung hạn)
    - K==25: LONG  (v5 best ở K=25)
    - K>=28: MED   (v9 dùng MED cho K dài, tốt hơn LONG)
    """
    if K <= 8:    return list(dict.fromkeys(FEATURES_SHORT))
    elif K <= 24: return list(dict.fromkeys(FEATURES_MED))
    elif K == 25: return list(dict.fromkeys(FEATURES_LONG))  # v5 strategy
    else:         return list(dict.fromkeys(FEATURES_MED))   # v9 strategy

print(f'SHORT:{len(get_base_features(1))} | MED:{len(get_base_features(12))} | '
      f'LONG(K=25):{len(get_base_features(25))} | MED(K=28+):{len(get_base_features(28))}')

Train: 25,481 | Val: 2,952 | Test: 4,152
SHORT:49 | MED:69 | LONG(K=25):56 | MED(K=28+):69


In [ ]:
# ─── CELL 5: SPEARMAN PRUNING ─────────────────────────────────────
from scipy.stats import spearmanr
import warnings, builtins

def prune_features(train_pdf, feature_cols, target_col='label',
                   min_target_corr=0.05, max_inter_corr=0.97):
    warnings.filterwarnings('ignore')
    X = train_pdf[feature_cols].fillna(0).values
    y = train_pdf[target_col].values

    keep = [c for i,c in enumerate(feature_cols)
            if builtins.abs(spearmanr(X[:,i], y)[0]) >= min_target_corr]
    if len(keep) < 5:
        keep = feature_cols[:]

    X2 = train_pdf[keep].fillna(0).values
    tc = {c: builtins.abs(spearmanr(X2[:,i], y)[0]) for i,c in enumerate(keep)}
    cm = np.corrcoef(X2.T)
    drop = set()
    for i in range(len(keep)):
        if keep[i] in drop: continue
        for j in range(i+1, len(keep)):
            if keep[j] in drop: continue
            if builtins.abs(cm[i,j]) > max_inter_corr:
                drop.add(keep[j] if tc[keep[i]] >= tc[keep[j]] else keep[i])

    return [c for c in keep if c not in drop]

print('✅ Helper ready')

✅ Helper ready


In [ ]:
# ─── CELL 6: TRAINING LOOP (v10 hybrid) ──────────────────────────
import builtins
os.makedirs('/content/models_v10', exist_ok=True)

HORIZONS  = list(range(1, 50, 3))
results   = []
eval_rmse = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='rmse')
eval_r2   = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='r2')

def get_rf_params(K):
    if K == 1:
        return dict(numTrees=300, maxDepth=8, minInstancesPerNode=10,
                    subsamplingRate=0.8, featureSubsetStrategy='sqrt')
    elif K <= 6:
        return dict(numTrees=250, maxDepth=7, minInstancesPerNode=8,
                    subsamplingRate=0.8, featureSubsetStrategy='sqrt')
    elif K <= 22:
        return dict(numTrees=200, maxDepth=7, minInstancesPerNode=10,
                    subsamplingRate=0.75, featureSubsetStrategy='sqrt')
    elif K == 25:
        return dict(numTrees=150, maxDepth=6, minInstancesPerNode=10,
                    subsamplingRate=0.75, featureSubsetStrategy='sqrt')
    else:
        return dict(numTrees=150, maxDepth=6, minInstancesPerNode=8,
                    subsamplingRate=0.75, featureSubsetStrategy='sqrt')

# Reference từ phân tích: best per K
BEST_REF = {
    1:8.76, 4:15.70, 7:19.41, 10:20.45, 13:20.76, 16:22.04,
    19:21.94, 22:21.36, 25:20.94, 28:21.80, 31:21.53, 34:21.80,
    37:22.40, 40:22.84, 43:21.91, 46:23.69, 49:23.01
}

for K in HORIZONS:
    print(f'\n🔵 Horizon = {K}h')

    df_k = df.withColumn('label_raw', lead('PM25', K).over(w_ts)) \
             .withColumn('label',     F.log1p(col('label_raw')))

    for c in WEATHER_COLS_LEAD:
        df_k = df_k.withColumn(f'{c}_atK', lead(c, K).over(w_ts))

    base_for_K = get_base_features(K)
    df_k = df_k \
        .withColumn('pm25_lagK',    lag('PM25', K).over(w_ts)) \
        .withColumn('horizon_norm', lit(float(K) / 49.0))

    ALL_FEATS = list(dict.fromkeys(base_for_K + WEATHER_AT_K + ['pm25_lagK','horizon_norm']))
    available = set(df_k.columns)
    ALL_FEATS = [f for f in ALL_FEATS if f in available]

    df_k = df_k.dropna(subset=ALL_FEATS + ['label','label_raw'])
    df_k = df_k.repartition(8).cache()
    df_k.count()

    train_df = df_k.filter(col('ts') < lit(CUTOFF_VAL))
    val_df   = df_k.filter((col('ts') >= lit(CUTOFF_VAL)) & (col('ts') < lit(CUTOFF_TEST)))
    test_df  = df_k.filter(col('ts') >= lit(CUTOFF_TEST))

    n_tr = train_df.count()
    if n_tr < 200:
        print(f'  ⚠️ Skip: chỉ có {n_tr} train rows')
        df_k.unpersist(); continue

    train_pdf = train_df.select(ALL_FEATS + ['label']).toPandas()
    pruned    = prune_features(train_pdf, ALL_FEATS)
    print(f'  Features: {len(ALL_FEATS)} -> {len(pruned)} after Spearman pruning')

    params = get_rf_params(K)

    asm1 = VectorAssembler(inputCols=pruned, outputCol='features', handleInvalid='skip')
    rf1  = RandomForestRegressor(featuresCol='features', labelCol='label',
                                  predictionCol='prediction', seed=42, **params)
    mdl1 = Pipeline(stages=[asm1, rf1]).fit(train_df)

    imps  = mdl1.stages[-1].featureImportances.toArray()
    imp_p = [f for f,imp in zip(pruned, imps) if imp >= 0.003]
    if len(imp_p) < 5: imp_p = pruned

    if len(imp_p) < len(pruned):
        print(f'  Importance pruning: {len(pruned)} -> {len(imp_p)}')
        asm2 = VectorAssembler(inputCols=imp_p, outputCol='features', handleInvalid='skip')
        rf2  = RandomForestRegressor(featuresCol='features', labelCol='label',
                                      predictionCol='prediction', seed=42, **params)
        model  = Pipeline(stages=[asm2, rf2]).fit(train_df)
        pruned = imp_p
    else:
        model = mdl1

    def decode(pred_df):
        return pred_df \
            .withColumn('prediction', F.expm1(col('prediction'))) \
            .withColumn('prediction', F.greatest(lit(0.0), col('prediction'))) \
            .withColumn('label', col('label_raw'))

    p_tr = decode(model.transform(train_df))
    p_va = decode(model.transform(val_df))
    p_te = decode(model.transform(test_df))

    rmse_tr = eval_rmse.evaluate(p_tr)
    rmse_va = eval_rmse.evaluate(p_va)
    rmse_te = eval_rmse.evaluate(p_te)
    r2_te   = eval_r2.evaluate(p_te)
    gap     = rmse_va - rmse_tr

    ref     = BEST_REF.get(K)
    vs_ref  = f'{rmse_te-ref:+.2f} vs best' if ref else ''
    flag_gap = '🟢' if gap < 6 else ('🟡' if gap < 10 else '🔴')
    flag_r2  = '🟢' if r2_te >= 0.85 else ('🟡' if r2_te >= 0.65 else '🔴')
    print(f'  Train RMSE: {rmse_tr:.2f} | Val RMSE: {rmse_va:.2f} '
          f'(gap={gap:.2f}{flag_gap}) | Test RMSE: {rmse_te:.2f} | R2: {r2_te:.3f} {flag_r2}  {vs_ref}')

    model.stages[-1].write().overwrite().save(f'/content/models_v10/gbt_k{K}')
    with open(f'/content/models_v10/features_k{K}.json','w') as fj:
        json.dump(pruned, fj)
    with open(f'/content/models_v10/meta_k{K}.json','w') as fj:
        json.dump({'K':K,'strategy':'LONG' if K==25 else ('SHORT' if K<=8 else 'MED'),
                   'n_features':len(pruned)}, fj)
    print(f'  💾 Saved K={K}')

    results.append({'K':K,'n_features':len(pruned),
                    'rmse_train':rmse_tr,'rmse_val':rmse_va,'gap':gap,
                    'rmse_test':rmse_te,'r2_test':r2_te})
    df_k.unpersist()

print('\n✅ Training done')


🔵 Horizon = 1h
  Features: 58 -> 44 after Spearman pruning
  Importance pruning: 44 -> 25
  Train RMSE: 6.87 | Val RMSE: 10.46 (gap=3.59🟢) | Test RMSE: 10.68 | R2: 0.888 🟢  +1.92 vs best
  💾 Saved K=1

🔵 Horizon = 4h
  Features: 58 -> 45 after Spearman pruning
  Importance pruning: 45 -> 33
  Train RMSE: 14.19 | Val RMSE: 21.44 (gap=7.25🟡) | Test RMSE: 20.32 | R2: 0.593 🔴  +4.62 vs best
  💾 Saved K=4

🔵 Horizon = 7h
  Features: 58 -> 45 after Spearman pruning
  Importance pruning: 45 -> 39
  Train RMSE: 16.67 | Val RMSE: 25.62 (gap=8.95🟡) | Test RMSE: 23.32 | R2: 0.464 🔴  +3.91 vs best
  💾 Saved K=7

🔵 Horizon = 10h
  Features: 78 -> 60 after Spearman pruning
  Importance pruning: 60 -> 48
  Train RMSE: 17.54 | Val RMSE: 25.74 (gap=8.19🟡) | Test RMSE: 23.16 | R2: 0.472 🔴  +2.71 vs best
  💾 Saved K=10

🔵 Horizon = 13h
  Features: 78 -> 63 after Spearman pruning
  Importance pruning: 63 -> 52
  Train RMSE: 17.69 | Val RMSE: 25.80 (gap=8.11🟡) | Test RMSE: 22.97 | R2: 0.480 🔴  +2.21 vs be

In [ ]:
# ─── CELL 7: RESULTS + SO SÁNH ────────────────────────────────────
res_df = pd.DataFrame(results).sort_values('K').reset_index(drop=True)

print('\n📊 RESULTS v10')
print(f'  {"K":>3}  {"feat":>5}  {"train":>7}  {"val":>7}  {"gap":>7}  {"test":>7}  {"R2":>6}')
for _, r in res_df.iterrows():
    fg = '🟢' if r.gap < 6 else ('🟡' if r.gap < 10 else '🔴')
    fr = '🟢' if r.r2_test >= 0.85 else ('🟡' if r.r2_test >= 0.65 else '🔴')
    print(f'  {int(r.K):>3}  {int(r.n_features):>5}  '
          f'{r.rmse_train:>7.2f}  {r.rmse_val:>7.2f}  {r.gap:>6.2f}{fg}  '
          f'{r.rmse_test:>7.2f}  {r.r2_test:>6.3f} {fr}')

print(f'\nOverall    — RMSE: {res_df.rmse_test.mean():.3f} | R2: {res_df.r2_test.mean():.3f} | Avg gap: {res_df.gap.mean():.2f}')
for label,lo,hi in [('Ngắn  (1-6h)',1,6),('Trung (7-24h)',7,24),('Dài  (25-49h)',25,49)]:
    s = res_df[res_df.K.between(lo,hi)]
    if len(s):
        print(f'{label} — RMSE: {s.rmse_test.mean():.3f} | R2: {s.r2_test.mean():.3f} | Avg gap: {s.gap.mean():.2f}')

best_ref = {1:8.76,4:15.70,7:19.41,10:20.45,13:20.76,16:22.04,
            19:21.94,22:21.36,25:20.94,28:21.80,31:21.53,34:21.80,
            37:22.40,40:22.84,43:21.91,46:23.69,49:23.01}
v4_ref   = {1:9.81,4:16.94,7:19.67,10:21.83,13:21.93,16:21.90,
            19:22.06,22:22.70,25:22.47,28:22.10,31:23.10,34:23.49,
            37:22.96,40:25.02,43:22.35,46:22.88,49:23.07}
v5_ref   = {1:9.27,4:16.48,7:19.74,10:21.42,13:22.96,16:22.24,
            19:21.94,22:22.67,25:20.94,28:21.83,31:22.76,34:23.44,
            37:23.42,40:22.84,43:21.91,46:24.14,49:25.26}

print('\n📊 SO SÁNH v4 | v5 | best_prev | v10')
print(f'  {"K":>3}  {"v4":>7}  {"v5":>7}  {"best":>7}  {"v10":>7}  {"Δv10-v4":>9}  {"Δv10-best":>10}')
wins_v4 = wins_best = 0
for _, r in res_df.iterrows():
    k = int(r.K)
    if k not in v4_ref: continue
    d4   = r.rmse_test - v4_ref[k]
    dbst = r.rmse_test - best_ref.get(k, 999)
    s4   = '✅' if d4   < 0 else '❌'
    sb   = '✅' if dbst < 0 else '❌'
    if d4   < 0: wins_v4   += 1
    if dbst < 0: wins_best += 1
    print(f'  {k:>3}  {v4_ref[k]:>7.2f}  {v5_ref[k]:>7.2f}  {best_ref.get(k,0):>7.2f}  '
          f'{r.rmse_test:>7.2f}  {d4:>+8.2f}{s4}  {dbst:>+9.2f}{sb}')

total = len(res_df)
print(f'\nv10 thắng v4: {wins_v4}/{total} | v10 thắng best_prev: {wins_best}/{total}')
res_df.to_csv('/content/results_v10.csv', index=False)
print('💾 /content/results_v10.csv')


📊 RESULTS v10
    K   feat    train      val      gap     test      R2
    1     25     6.87    10.46    3.59🟢    10.68   0.888 🟢
    4     33    14.19    21.44    7.25🟡    20.32   0.593 🔴
    7     39    16.67    25.62    8.95🟡    23.32   0.464 🔴
   10     48    17.54    25.74    8.19🟡    23.16   0.472 🔴
   13     52    17.69    25.80    8.11🟡    22.97   0.480 🔴
   16     49    17.98    25.59    7.60🟡    23.33   0.464 🔴
   19     50    17.66    24.95    7.29🟡    22.71   0.492 🔴
   22     48    17.86    25.16    7.30🟡    23.00   0.479 🔴
   25     40    18.95    25.66    6.71🟡    23.99   0.434 🔴
   28     43    19.44    26.96    7.52🟡    24.68   0.400 🔴
   31     44    19.70    27.77    8.08🟡    25.06   0.382 🔴
   34     46    19.84    28.03    8.19🟡    25.35   0.367 🔴
   37     48    19.99    28.26    8.27🟡    25.44   0.363 🔴
   40     45    19.88    28.32    8.44🟡    25.15   0.378 🔴
   43     40    19.67    27.38    7.70🟡    24.48   0.410 🔴
   46     44    20.06    27.68    7.62🟡    

In [ ]:
# ─── CELL 8: ZIP & DOWNLOAD ──────────────────────────────────────
!zip -r /content/models_v10.zip /content/models_v10 /content/results_v10.csv -q
from google.colab import files
files.download('/content/models_v10.zip')
files.download('/content/results_v10.csv')
print('✅ Done')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done
